In [1]:
# standardize core interaction schema for baseline models

# update paths if necessary
FILE_PATH = r"D:\256_group_dataset\HotelRec.txt"
DB_PATH = r"D:\256_group_dataset\hotelrec_preprocessing.duckdb"

RAW_TABLE = "hotelrec_raw"
CORE_TABLE = "interactions_core_stage1"

In [2]:
import os
import duckdb
import pandas as pd

# Create DuckDB temp directory so large operations do not stay in memory
tmp_dir = os.path.join(os.path.dirname(DB_PATH), "duckdb_tmp")
os.makedirs(tmp_dir, exist_ok=True)

con = duckdb.connect(DB_PATH)
con.execute("PRAGMA threads=4;")
con.execute(f"SET temp_directory='{tmp_dir}';")

safe_path = FILE_PATH.replace("\\", "/").replace("'", "''")

print("Connected to:", DB_PATH)
print("Reading from :", FILE_PATH)

Connected to: D:\256_group_dataset\hotelrec_preprocessing.duckdb
Reading from : D:\256_group_dataset\HotelRec.txt


In [3]:
# Load the raw HotelRec file into DuckDB
# This keeps the original columns available for later sections.

con.execute(f"DROP TABLE IF EXISTS {RAW_TABLE}")

con.execute(f"""
CREATE TABLE {RAW_TABLE} AS
SELECT *
FROM read_json(
    '{safe_path}',
    columns = {{
        author: 'VARCHAR',
        hotel_url: 'VARCHAR',
        date: 'TIMESTAMP',
        rating: 'DOUBLE',
        title: 'VARCHAR',
        text: 'VARCHAR',
        property_dict: 'STRUCT("sleep quality" DOUBLE, value DOUBLE, rooms DOUBLE, service DOUBLE, cleanliness DOUBLE, location DOUBLE)'
    }},
    format = 'auto',
    ignore_errors = true
)
""")

print(f"Created raw table: {RAW_TABLE}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Created raw table: hotelrec_raw


In [4]:
# Create the standardized core interaction table used by baseline models

con.execute(f"DROP TABLE IF EXISTS {CORE_TABLE}")

con.execute(f"""
CREATE TABLE {CORE_TABLE} AS
SELECT
    ROW_NUMBER() OVER () AS interaction_id,
    CAST(author AS VARCHAR)      AS user_id,
    CAST(hotel_url AS VARCHAR)   AS item_id,
    CAST(rating AS DOUBLE)       AS rating,
    CAST(date AS TIMESTAMP)      AS timestamp
FROM {RAW_TABLE}
""")

print(f"Created core table: {CORE_TABLE}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Created core table: interactions_core_stage1


In [5]:
# Quick schema check
schema_df = con.execute(f"DESCRIBE {CORE_TABLE}").fetchdf()
display(schema_df)

,column_name,column_type,null,key,default,extra
0,interaction_id,BIGINT,YES,None,None,None
1,user_id,VARCHAR,YES,None,None,None
2,item_id,VARCHAR,YES,None,None,None
3,rating,DOUBLE,YES,None,None,None
4,timestamp,TIMESTAMP,YES,None,None,None


In [6]:
# Core audit summary: size, uniqueness, nulls, and rating range
audit_df = con.execute(f"""
SELECT
    COUNT(*) AS n_interactions,
    COUNT(DISTINCT user_id) AS n_users,
    COUNT(DISTINCT item_id) AS n_items,
    COUNT(CASE WHEN user_id IS NULL OR TRIM(user_id) = '' THEN 1 END) AS null_user_id,
    COUNT(CASE WHEN item_id IS NULL OR TRIM(item_id) = '' THEN 1 END) AS null_item_id,
    COUNT(CASE WHEN rating IS NULL THEN 1 END) AS null_rating,
    COUNT(CASE WHEN timestamp IS NULL THEN 1 END) AS null_timestamp,
    MIN(rating) AS min_rating,
    MAX(rating) AS max_rating,
    MIN(timestamp) AS min_timestamp,
    MAX(timestamp) AS max_timestamp
FROM {CORE_TABLE}
""").fetchdf()

display(audit_df)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n_interactions,n_users,n_items,null_user_id,null_item_id,null_rating,null_timestamp,min_rating,max_rating,min_timestamp,max_timestamp
0,50264531,21891404,365057,0,0,0,0,1.0,5.0,2001-02-01,2019-09-20


In [7]:
# Remove invalid placeholder users and invalid core records

INVALID_CORE_TABLE = "interactions_invalid_stage2"
CLEAN_STAGE2_TABLE = "interactions_clean_stage2"

# Add or remove placeholder IDs here if needed
PLACEHOLDER_USER_IDS = [
    "/undefined",
    "undefined",
    "null",
    "none",
    "nan"
]

import pandas as pd

placeholder_sql = ", ".join([f"'{x.lower()}'" for x in PLACEHOLDER_USER_IDS])
print("Placeholder user IDs being removed:", PLACEHOLDER_USER_IDS)

Placeholder user IDs being removed: ['/undefined', 'undefined', 'null', 'none', 'nan']


In [8]:
# Create a table of rows that fail the Section 2 validity rules
con.execute(f"DROP TABLE IF EXISTS {INVALID_CORE_TABLE}")

con.execute(f"""
CREATE TABLE {INVALID_CORE_TABLE} AS
SELECT
    *,
    CASE
        WHEN user_id IS NULL OR TRIM(user_id) = '' THEN 'missing_user_id'
        WHEN LOWER(TRIM(user_id)) IN ({placeholder_sql}) THEN 'placeholder_user_id'
        WHEN item_id IS NULL OR TRIM(item_id) = '' THEN 'missing_item_id'
        WHEN rating IS NULL THEN 'missing_rating'
        WHEN rating < 1 OR rating > 5 THEN 'rating_out_of_range'
        WHEN timestamp IS NULL THEN 'missing_timestamp'
        ELSE NULL
    END AS removal_reason
FROM {CORE_TABLE}
WHERE
    user_id IS NULL
    OR TRIM(user_id) = ''
    OR LOWER(TRIM(user_id)) IN ({placeholder_sql})
    OR item_id IS NULL
    OR TRIM(item_id) = ''
    OR rating IS NULL
    OR rating < 1
    OR rating > 5
    OR timestamp IS NULL
""")

print(f"Created invalid-row table: {INVALID_CORE_TABLE}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Created invalid-row table: interactions_invalid_stage2


In [9]:
# Audit what will be removed
invalid_summary_df = con.execute(f"""
SELECT
    removal_reason,
    COUNT(*) AS n_rows,
    COUNT(DISTINCT user_id) AS n_users,
    COUNT(DISTINCT item_id) AS n_items
FROM {INVALID_CORE_TABLE}
GROUP BY removal_reason
ORDER BY n_rows DESC
""").fetchdf()

display(invalid_summary_df)

placeholder_users_df = con.execute(f"""
SELECT
    user_id,
    COUNT(*) AS n_rows,
    MIN(timestamp) AS first_timestamp,
    MAX(timestamp) AS last_timestamp
FROM {INVALID_CORE_TABLE}
WHERE removal_reason = 'placeholder_user_id'
GROUP BY user_id
ORDER BY n_rows DESC
""").fetchdf()

display(placeholder_users_df)

,removal_reason,n_rows,n_users,n_items
0,placeholder_user_id,1216923,4,52018


,user_id,n_rows,first_timestamp,last_timestamp
0,/undefined,1193017,2001-02-01,2019-05-14 00:06:23
1,null,23889,2002-11-01,2019-05-02 21:00:55
2,Undefined,12,2010-06-01,2017-12-01 00:00:00
3,none,5,2005-02-01,2009-08-01 00:00:00


In [10]:
# Create the cleaned Stage 2 table by removing invalid rows
con.execute(f"DROP TABLE IF EXISTS {CLEAN_STAGE2_TABLE}")

con.execute(f"""
CREATE TABLE {CLEAN_STAGE2_TABLE} AS
SELECT c.*
FROM {CORE_TABLE} c
LEFT JOIN {INVALID_CORE_TABLE} bad
    ON c.interaction_id = bad.interaction_id
WHERE bad.interaction_id IS NULL
""")

print(f"Created cleaned table: {CLEAN_STAGE2_TABLE}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Created cleaned table: interactions_clean_stage2


In [11]:
# Compare before vs after Section 2
section2_compare_df = con.execute(f"""
SELECT 'before_section2' AS dataset,
       COUNT(*) AS n_interactions,
       COUNT(DISTINCT user_id) AS n_users,
       COUNT(DISTINCT item_id) AS n_items
FROM {CORE_TABLE}

UNION ALL

SELECT 'after_section2' AS dataset,
       COUNT(*) AS n_interactions,
       COUNT(DISTINCT user_id) AS n_users,
       COUNT(DISTINCT item_id) AS n_items
FROM {CLEAN_STAGE2_TABLE}
""").fetchdf()

display(section2_compare_df)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,dataset,n_interactions,n_users,n_items
0,before_section2,50264531,21891404,365057
1,after_section2,49047608,21891400,365055


In [12]:
# Sanity check: confirm no invalid core rows remain
post_check_df = con.execute(f"""
SELECT
    COUNT(CASE WHEN user_id IS NULL OR TRIM(user_id) = '' THEN 1 END) AS bad_user_id,
    COUNT(CASE WHEN LOWER(TRIM(user_id)) IN ({placeholder_sql}) THEN 1 END) AS placeholder_user_id,
    COUNT(CASE WHEN item_id IS NULL OR TRIM(item_id) = '' THEN 1 END) AS bad_item_id,
    COUNT(CASE WHEN rating IS NULL OR rating < 1 OR rating > 5 THEN 1 END) AS bad_rating,
    COUNT(CASE WHEN timestamp IS NULL THEN 1 END) AS bad_timestamp
FROM {CLEAN_STAGE2_TABLE}
""").fetchdf()

display(post_check_df)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,bad_user_id,placeholder_user_id,bad_item_id,bad_rating,bad_timestamp
0,0,0,0,0,0


In [13]:
# Separate repeated user–hotel pairs into their own table,
# then create a deduplicated modeling table by keeping the latest interaction.

DUP_GROUPS_TABLE = "duplicate_user_item_groups_stage3"
DUP_ROWS_TABLE = "duplicate_user_item_rows_stage3"
DEDUP_STAGE3_TABLE = "interactions_dedup_stage3"

In [14]:
# Find duplicate user-item groups (pairs that appear more than once)
con.execute(f"DROP TABLE IF EXISTS {DUP_GROUPS_TABLE}")

con.execute(f"""
CREATE TABLE {DUP_GROUPS_TABLE} AS
SELECT
    user_id,
    item_id,
    COUNT(*) AS pair_count,
    MIN(timestamp) AS first_timestamp,
    MAX(timestamp) AS last_timestamp,
    MIN(rating) AS min_rating,
    MAX(rating) AS max_rating
FROM {CLEAN_STAGE2_TABLE}
GROUP BY user_id, item_id
HAVING COUNT(*) > 1
""")

print(f"Created duplicate-group table: {DUP_GROUPS_TABLE}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Created duplicate-group table: duplicate_user_item_groups_stage3


In [15]:
# Save ALL rows belonging to duplicate user-item pairs for future use
con.execute(f"DROP TABLE IF EXISTS {DUP_ROWS_TABLE}")

con.execute(f"""
CREATE TABLE {DUP_ROWS_TABLE} AS
SELECT c.*
FROM {CLEAN_STAGE2_TABLE} c
INNER JOIN {DUP_GROUPS_TABLE} d
    ON c.user_id = d.user_id
   AND c.item_id = d.item_id
""")

print(f"Created duplicate-rows table: {DUP_ROWS_TABLE}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Created duplicate-rows table: duplicate_user_item_rows_stage3


In [16]:
# Quick audit of duplicates
dup_summary_df = con.execute(f"""
SELECT
    COUNT(*) AS n_duplicate_pairs,
    SUM(pair_count) AS n_rows_in_duplicate_pairs,
    COUNT(DISTINCT user_id) AS n_users_involved,
    COUNT(DISTINCT item_id) AS n_items_involved,
    AVG(pair_count) AS avg_rows_per_duplicate_pair,
    MAX(pair_count) AS max_rows_for_single_pair
FROM {DUP_GROUPS_TABLE}
""").fetchdf()

display(dup_summary_df)

,n_duplicate_pairs,n_rows_in_duplicate_pairs,n_users_involved,n_items_involved,avg_rows_per_duplicate_pair,max_rows_for_single_pair
0,1040994,2352825.0,832603,158570,2.260172,23


In [17]:
# Distribution of how many times the same user-item pair repeats
dup_pair_dist_df = con.execute(f"""
SELECT
    pair_count,
    COUNT(*) AS n_pairs
FROM {DUP_GROUPS_TABLE}
GROUP BY pair_count
ORDER BY pair_count
""").fetchdf()

display(dup_pair_dist_df.head(20))

,pair_count,n_pairs
0,2,865546
1,3,121962
2,4,32265
3,5,11551
4,6,4827
5,7,2236
6,8,1174
7,9,608
8,10,321
9,11,212


In [18]:
# Build the deduplicated modeling table:
# keep the latest interaction for each (user_id, item_id) pair.
# Tiebreaker: if timestamps are identical, keep the row with the largest interaction_id.

con.execute(f"DROP TABLE IF EXISTS {DEDUP_STAGE3_TABLE}")

con.execute(f"""
CREATE TABLE {DEDUP_STAGE3_TABLE} AS
WITH ranked AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY user_id, item_id
            ORDER BY timestamp DESC, interaction_id DESC
        ) AS rn
    FROM {CLEAN_STAGE2_TABLE}
)
SELECT
    interaction_id,
    user_id,
    item_id,
    rating,
    timestamp
FROM ranked
WHERE rn = 1
""")

print(f"Created deduplicated table: {DEDUP_STAGE3_TABLE}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Created deduplicated table: interactions_dedup_stage3


In [19]:
# Before vs after Section 3
section3_compare_df = con.execute(f"""
SELECT 'before_section3' AS dataset,
       COUNT(*) AS n_interactions,
       COUNT(DISTINCT user_id) AS n_users,
       COUNT(DISTINCT item_id) AS n_items,
       COUNT(*) - COUNT(DISTINCT (user_id, item_id)) AS extra_rows_from_duplicates
FROM {CLEAN_STAGE2_TABLE}

UNION ALL

SELECT 'after_section3' AS dataset,
       COUNT(*) AS n_interactions,
       COUNT(DISTINCT user_id) AS n_users,
       COUNT(DISTINCT item_id) AS n_items,
       COUNT(*) - COUNT(DISTINCT (user_id, item_id)) AS extra_rows_from_duplicates
FROM {DEDUP_STAGE3_TABLE}
""").fetchdf()

display(section3_compare_df)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,dataset,n_interactions,n_users,n_items,extra_rows_from_duplicates
0,before_section3,49047608,21891400,365055,1311831
1,after_section3,47735777,21891400,365055,0


In [20]:
# Sanity check: confirm no duplicate user-item pairs remain
post_dedup_check_df = con.execute(f"""
SELECT
    COUNT(*) AS remaining_duplicate_pairs
FROM (
    SELECT user_id, item_id
    FROM {DEDUP_STAGE3_TABLE}
    GROUP BY user_id, item_id
    HAVING COUNT(*) > 1
) t
""").fetchdf()

display(post_dedup_check_df)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,remaining_duplicate_pairs
0,0


In [21]:
# Apply a iterative 5-core filter

K_CORE_K = 5

KCORE_SOURCE_TABLE = DEDUP_STAGE3_TABLE
KCORE_WORK_TABLE = "interactions_kcore_work_stage5"
KCORE_NEXT_TABLE = "interactions_kcore_next_stage5"
KCORE_FINAL_TABLE = "interactions_5core_stage5"

In [22]:
# Helper: get compact dataset stats
def get_table_stats(con, table_name):
    query = f"""
    SELECT
        COUNT(*) AS n_interactions,
        COUNT(DISTINCT user_id) AS n_users,
        COUNT(DISTINCT item_id) AS n_items
    FROM {table_name}
    """
    row = con.execute(query).fetchdf().iloc[0]
    
    n_interactions = int(row["n_interactions"])
    n_users = int(row["n_users"])
    n_items = int(row["n_items"])
    
    if n_users == 0 or n_items == 0:
        sparsity_pct = None
    else:
        density = n_interactions / (n_users * n_items)
        sparsity_pct = 100 * (1 - density)
    
    return {
        "n_interactions": n_interactions,
        "n_users": n_users,
        "n_items": n_items,
        "sparsity_pct": sparsity_pct
    }

In [23]:
# Starting point for iterative filtering
con.execute(f"DROP TABLE IF EXISTS {KCORE_WORK_TABLE}")
con.execute(f"""
CREATE TABLE {KCORE_WORK_TABLE} AS
SELECT *
FROM {KCORE_SOURCE_TABLE}
""")

initial_stats = get_table_stats(con, KCORE_WORK_TABLE)
print("Initial stats:")
print(initial_stats)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Initial stats:
{'n_interactions': 47735777, 'n_users': 21891400, 'n_items': 365055, 'sparsity_pct': 99.99940267298284}


In [24]:
# iterative k-core:
# repeatedly remove users/items with fewer than K interactions until stable

iteration_logs = []
iteration = 0

while True:
    iteration += 1
    
    before_stats = get_table_stats(con, KCORE_WORK_TABLE)
    
    con.execute(f"DROP TABLE IF EXISTS {KCORE_NEXT_TABLE}")
    con.execute(f"""
    CREATE TABLE {KCORE_NEXT_TABLE} AS
    SELECT w.*
    FROM {KCORE_WORK_TABLE} w
    INNER JOIN (
        SELECT user_id
        FROM {KCORE_WORK_TABLE}
        GROUP BY user_id
        HAVING COUNT(*) >= {K_CORE_K}
    ) u
        ON w.user_id = u.user_id
    INNER JOIN (
        SELECT item_id
        FROM {KCORE_WORK_TABLE}
        GROUP BY item_id
        HAVING COUNT(*) >= {K_CORE_K}
    ) i
        ON w.item_id = i.item_id
    """)
    
    after_stats = get_table_stats(con, KCORE_NEXT_TABLE)
    
    iteration_logs.append({
        "iteration": iteration,
        "before_n_interactions": before_stats["n_interactions"],
        "after_n_interactions": after_stats["n_interactions"],
        "before_n_users": before_stats["n_users"],
        "after_n_users": after_stats["n_users"],
        "before_n_items": before_stats["n_items"],
        "after_n_items": after_stats["n_items"],
        "rows_removed": before_stats["n_interactions"] - after_stats["n_interactions"]
    })
    
    # Stable = no more rows removed
    if after_stats["n_interactions"] == before_stats["n_interactions"]:
        break
    
    # Replace work table with next iteration table
    con.execute(f"DROP TABLE IF EXISTS {KCORE_WORK_TABLE}")
    con.execute(f"ALTER TABLE {KCORE_NEXT_TABLE} RENAME TO {KCORE_WORK_TABLE}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [25]:
# Save the final stable 5-core table
con.execute(f"DROP TABLE IF EXISTS {KCORE_FINAL_TABLE}")
con.execute(f"""
CREATE TABLE {KCORE_FINAL_TABLE} AS
SELECT *
FROM {KCORE_NEXT_TABLE}
""")

print(f"Created final 5-core table: {KCORE_FINAL_TABLE}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Created final 5-core table: interactions_5core_stage5


In [26]:
# Iteration log
kcore_iteration_df = pd.DataFrame(iteration_logs)
display(kcore_iteration_df)

,iteration,before_n_interactions,after_n_interactions,before_n_users,after_n_users,before_n_items,after_n_items,rows_removed
0,1,47735777,20146765,21891400,1946900,365055,362178,27589012
1,2,20146765,20000403,1946900,1946873,362178,311107,146362
2,3,20000403,19927713,1946873,1928410,311107,311106,72690
3,4,19927713,19923128,1928410,1928410,311106,309946,4585
4,5,19923128,19920896,1928410,1927852,309946,309946,2232
5,6,19920896,19920744,1927852,1927852,309946,309908,152
6,7,19920744,19920684,1927852,1927837,309908,309908,60
7,8,19920684,19920672,1927837,1927837,309908,309905,12
8,9,19920672,19920664,1927837,1927835,309905,309905,8
9,10,19920664,19920660,1927835,1927835,309905,309904,4


In [27]:
# Before vs after 5-core
before_after_5core_df = pd.DataFrame([
    {
        "dataset": "before_5core",
        **get_table_stats(con, KCORE_SOURCE_TABLE)
    },
    {
        "dataset": "after_5core",
        **get_table_stats(con, KCORE_FINAL_TABLE)
    }
])

display(before_after_5core_df)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,dataset,n_interactions,n_users,n_items,sparsity_pct
0,before_5core,47735777,21891400,365055,99.999403
1,after_5core,19920656,1927834,309904,99.996666


In [28]:
# Sanity checks:
# every remaining user and item must have at least K_CORE_K interactions

kcore_sanity_df = con.execute(f"""
SELECT
    MIN(user_cnt) AS min_user_interactions,
    MAX(user_cnt) AS max_user_interactions
FROM (
    SELECT user_id, COUNT(*) AS user_cnt
    FROM {KCORE_FINAL_TABLE}
    GROUP BY user_id
) u
""").fetchdf()

kcore_item_sanity_df = con.execute(f"""
SELECT
    MIN(item_cnt) AS min_item_interactions,
    MAX(item_cnt) AS max_item_interactions
FROM (
    SELECT item_id, COUNT(*) AS item_cnt
    FROM {KCORE_FINAL_TABLE}
    GROUP BY item_id
) i
""").fetchdf()

print("User-count sanity check")
display(kcore_sanity_df)

print("Item-count sanity check")
display(kcore_item_sanity_df)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

User-count sanity check


,min_user_interactions,max_user_interactions
0,5,580


Item-count sanity check


,min_item_interactions,max_item_interactions
0,5,5150


In [29]:
# Extra audit: retention percentages
source_stats = get_table_stats(con, KCORE_SOURCE_TABLE)
final_stats = get_table_stats(con, KCORE_FINAL_TABLE)

retention_df = pd.DataFrame([{
    "k_core_k": K_CORE_K,
    "interaction_retention_pct": round(100 * final_stats["n_interactions"] / source_stats["n_interactions"], 4) if source_stats["n_interactions"] else None,
    "user_retention_pct": round(100 * final_stats["n_users"] / source_stats["n_users"], 4) if source_stats["n_users"] else None,
    "item_retention_pct": round(100 * final_stats["n_items"] / source_stats["n_items"], 4) if source_stats["n_items"] else None
}])

display(retention_df)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,k_core_k,interaction_retention_pct,user_retention_pct,item_retention_pct
0,5,41.7311,8.8064,84.8924


In [30]:
# Per-user leave-last-out temporal split
# For each user, their most recent interaction → test,
# second most recent → valid, all others → train.
# 5-core guarantees ≥5 interactions per user, so every user
# gets ≥3 train, 1 valid, 1 test.

SPLIT_SOURCE_TABLE = KCORE_FINAL_TABLE
SPLIT_TABLE = "interactions_5core_split_stage6"

In [31]:
con.execute(f"DROP TABLE IF EXISTS {SPLIT_TABLE}")

con.execute(f"""
CREATE TABLE {SPLIT_TABLE} AS
WITH ranked AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY user_id
            ORDER BY timestamp DESC, interaction_id DESC
        ) AS rn
    FROM {SPLIT_SOURCE_TABLE}
)
SELECT
    interaction_id,
    user_id,
    item_id,
    rating,
    timestamp,
    CASE
        WHEN rn = 1 THEN 'test'
        WHEN rn = 2 THEN 'valid'
        ELSE 'train'
    END AS split
FROM ranked
""")

print(f"Created split table: {SPLIT_TABLE}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Created split table: interactions_5core_split_stage6


In [32]:
split_size_df = con.execute(f"""
SELECT
    split,
    COUNT(*) AS n_interactions,
    COUNT(DISTINCT user_id) AS n_users,
    COUNT(DISTINCT item_id) AS n_items,
    MIN(timestamp) AS min_timestamp,
    MAX(timestamp) AS max_timestamp
FROM {SPLIT_TABLE}
GROUP BY split
ORDER BY CASE split
    WHEN 'train' THEN 1
    WHEN 'valid' THEN 2
    WHEN 'test' THEN 3
END
""").fetchdf()

display(split_size_df)

# Show proportions
total = split_size_df["n_interactions"].sum()
split_size_df["pct"] = (100 * split_size_df["n_interactions"] / total).round(2)
print("\nSplit proportions:")
for _, row in split_size_df.iterrows():
    print(f"  {row['split']:>5}: {row['n_interactions']:>12,} interactions  ({row['pct']}%)")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,split,n_interactions,n_users,n_items,min_timestamp,max_timestamp
0,train,16064988,1927834,309395,2002-07-01,2019-05-08
1,valid,1927834,1927834,253644,2002-10-01,2019-05-11
2,test,1927834,1927834,247137,2002-10-01,2019-09-20



Split proportions:
  train:   16,064,988 interactions  (80.64%)
  valid:    1,927,834 interactions  (9.68%)
   test:    1,927,834 interactions  (9.68%)


In [33]:
# This is the key guarantee of leave-last-out:
# every user must have exactly 1 test, exactly 1 valid, and ≥3 train interactions.

user_split_audit_df = con.execute(f"""
SELECT
    MIN(train_count) AS min_train_per_user,
    MAX(train_count) AS max_train_per_user,
    MIN(valid_count) AS min_valid_per_user,
    MAX(valid_count) AS max_valid_per_user,
    MIN(test_count) AS min_test_per_user,
    MAX(test_count) AS max_test_per_user,
    COUNT(*) AS total_users
FROM (
    SELECT
        user_id,
        COUNT(CASE WHEN split = 'train' THEN 1 END) AS train_count,
        COUNT(CASE WHEN split = 'valid' THEN 1 END) AS valid_count,
        COUNT(CASE WHEN split = 'test' THEN 1 END) AS test_count
    FROM {SPLIT_TABLE}
    GROUP BY user_id
) per_user
""").fetchdf()

display(user_split_audit_df)

# Confirm the guarantees hold
row = user_split_audit_df.iloc[0]
assert row["min_train_per_user"] >= 3, f"Some user has only {row['min_train_per_user']} train interactions"
assert row["min_valid_per_user"] == 1 and row["max_valid_per_user"] == 1, "Valid count per user should be exactly 1"
assert row["min_test_per_user"] == 1 and row["max_test_per_user"] == 1, "Test count per user should be exactly 1"
print("✓ Every user has exactly 1 test, exactly 1 valid, and ≥3 train interactions")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,min_train_per_user,max_train_per_user,min_valid_per_user,max_valid_per_user,min_test_per_user,max_test_per_user,total_users
0,3,578,1,1,1,1,1927834


✓ Every user has exactly 1 test, exactly 1 valid, and ≥3 train interactions


In [34]:
# With leave-last-out, every USER is guaranteed to appear in train.
# But some ITEMS may only appear in val/test if all their reviewers
# happened to review that item as their last or second-to-last interaction.

unseen_items_df = con.execute(f"""
WITH train_items AS (
    SELECT DISTINCT item_id FROM {SPLIT_TABLE} WHERE split = 'train'
)
SELECT
    s.split,
    COUNT(*) AS n_interactions,
    COUNT(CASE WHEN ti.item_id IS NULL THEN 1 END) AS unseen_item_interactions,
    ROUND(100.0 * COUNT(CASE WHEN ti.item_id IS NULL THEN 1 END) / COUNT(*), 4) AS unseen_item_pct
FROM {SPLIT_TABLE} s
LEFT JOIN train_items ti ON s.item_id = ti.item_id
WHERE s.split IN ('valid', 'test')
GROUP BY s.split
ORDER BY CASE s.split WHEN 'valid' THEN 1 WHEN 'test' THEN 2 END
""").fetchdf()

display(unseen_items_df)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,split,n_interactions,unseen_item_interactions,unseen_item_pct
0,valid,1927834,997,0.0517
1,test,1927834,2458,0.1275


In [35]:
# Verify that for each user, test timestamp ≥ valid timestamp ≥ max train timestamp.
# Small number of ties are acceptable (same-day reviews), but reversals would be a bug.

temporal_check_df = con.execute(f"""
WITH user_splits AS (
    SELECT
        user_id,
        MAX(CASE WHEN split = 'train' THEN timestamp END) AS max_train_ts,
        MAX(CASE WHEN split = 'valid' THEN timestamp END) AS valid_ts,
        MAX(CASE WHEN split = 'test'  THEN timestamp END) AS test_ts
    FROM {SPLIT_TABLE}
    GROUP BY user_id
)
SELECT
    COUNT(*) AS total_users,
    COUNT(CASE WHEN test_ts >= valid_ts THEN 1 END) AS test_after_valid,
    COUNT(CASE WHEN valid_ts >= max_train_ts THEN 1 END) AS valid_after_train,
    COUNT(CASE WHEN test_ts >= valid_ts AND valid_ts >= max_train_ts THEN 1 END) AS fully_ordered
FROM user_splits
""").fetchdf()

display(temporal_check_df)

row = temporal_check_df.iloc[0]
print(f"✓ {row['fully_ordered']}/{row['total_users']} users have test ≥ valid ≥ train timestamps "
      f"({100*row['fully_ordered']/row['total_users']:.2f}%)")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_users,test_after_valid,valid_after_train,fully_ordered
0,1927834,1927834,1927834,1927834


✓ 1927834/1927834 users have test ≥ valid ≥ train timestamps (100.00%)


## Stage 8 — Integer ID Mapping

Map string user_id and item_id to contiguous integers (0 to N−1).
Required for sparse matrix construction, embedding tables, and library compatibility.

In [36]:
# Create integer-mapped table with original IDs preserved for reference

ID_MAP_TABLE = "interactions_id_mapped_stage8"

con.execute(f"DROP TABLE IF EXISTS {ID_MAP_TABLE}")

con.execute(f"""
CREATE TABLE {ID_MAP_TABLE} AS
WITH user_map AS (
    SELECT
        user_id AS original_user_id,
        DENSE_RANK() OVER (ORDER BY user_id) - 1 AS user_idx
    FROM (SELECT DISTINCT user_id FROM {SPLIT_TABLE})
),
item_map AS (
    SELECT
        item_id AS original_item_id,
        DENSE_RANK() OVER (ORDER BY item_id) - 1 AS item_idx
    FROM (SELECT DISTINCT item_id FROM {SPLIT_TABLE})
)
SELECT
    s.interaction_id,
    um.user_idx,
    im.item_idx,
    s.rating,
    s.timestamp,
    s.split,
    s.user_id  AS original_user_id,
    s.item_id  AS original_item_id
FROM {SPLIT_TABLE} s
JOIN user_map um ON s.user_id = um.original_user_id
JOIN item_map im ON s.item_id = im.original_item_id
""")

print(f"Created ID-mapped table: {ID_MAP_TABLE}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Created ID-mapped table: interactions_id_mapped_stage8


In [37]:
# Verify: integer IDs are contiguous 0..N-1, cardinality matches, no nulls

id_map_audit_df = con.execute(f"""
SELECT
    COUNT(DISTINCT user_idx)  AS n_user_ids,
    MIN(user_idx)             AS min_user_idx,
    MAX(user_idx)             AS max_user_idx,
    COUNT(DISTINCT item_idx)  AS n_item_ids,
    MIN(item_idx)             AS min_item_idx,
    MAX(item_idx)             AS max_item_idx,
    COUNT(CASE WHEN user_idx IS NULL THEN 1 END) AS null_user_idx,
    COUNT(CASE WHEN item_idx IS NULL THEN 1 END) AS null_item_idx
FROM {ID_MAP_TABLE}
""").fetchdf()

display(id_map_audit_df)

row = id_map_audit_df.iloc[0]
assert row["min_user_idx"] == 0, "User IDs should start at 0"
assert row["max_user_idx"] == row["n_user_ids"] - 1, "User IDs should be contiguous"
assert row["min_item_idx"] == 0, "Item IDs should start at 0"
assert row["max_item_idx"] == row["n_item_ids"] - 1, "Item IDs should be contiguous"
assert row["null_user_idx"] == 0, "No null user IDs"
assert row["null_item_idx"] == 0, "No null item IDs"
print(f"✓ user_idx: 0..{row['max_user_idx']:,}  ({row['n_user_ids']:,} users)")
print(f"✓ item_idx: 0..{row['max_item_idx']:,}  ({row['n_item_ids']:,} items)")

,n_user_ids,min_user_idx,max_user_idx,n_item_ids,min_item_idx,max_item_idx,null_user_idx,null_item_idx
0,1927834,0,1927833,309904,0,309903,0,0


✓ user_idx: 0..1,927,833  (1,927,834 users)
✓ item_idx: 0..309,903  (309,904 items)


## Stage 9 — Bias Terms & Popularity Metrics

Computed from **training data only** to prevent leakage.
- Global mean rating (μ)
- Per-user bias: b_u = mean(user's train ratings) − μ
- Per-item bias: b_i = mean(item's train ratings) − μ
- Bayesian/damped average per item for the popularity baseline

In [38]:
# Compute global mean from training split only

GLOBAL_MEAN_RESULT = con.execute(f"""
    SELECT AVG(rating) AS global_mean
    FROM {ID_MAP_TABLE}
    WHERE split = 'train'
""").fetchdf()

GLOBAL_MEAN = float(GLOBAL_MEAN_RESULT["global_mean"].iloc[0])
print(f"Global mean rating (train only): {GLOBAL_MEAN:.6f}")

Global mean rating (train only): 4.071144


In [39]:
# Per-user bias from training data

USER_BIAS_TABLE = "user_bias_stage9"

con.execute(f"DROP TABLE IF EXISTS {USER_BIAS_TABLE}")

con.execute(f"""
CREATE TABLE {USER_BIAS_TABLE} AS
SELECT
    user_idx,
    original_user_id,
    COUNT(*) AS train_review_count,
    AVG(rating) AS train_mean_rating,
    AVG(rating) - {GLOBAL_MEAN} AS user_bias
FROM {ID_MAP_TABLE}
WHERE split = 'train'
GROUP BY user_idx, original_user_id
""")

print(f"Created user bias table: {USER_BIAS_TABLE}")

user_bias_stats = con.execute(f"""
SELECT
    COUNT(*) AS n_users,
    AVG(user_bias) AS mean_bias,
    STDDEV(user_bias) AS std_bias,
    MIN(user_bias) AS min_bias,
    MAX(user_bias) AS max_bias,
    PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY user_bias) AS p25,
    PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY user_bias) AS p50,
    PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY user_bias) AS p75
FROM {USER_BIAS_TABLE}
""").fetchdf()

display(user_bias_stats)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Created user bias table: user_bias_stage9


,n_users,mean_bias,std_bias,min_bias,max_bias,p25,p50,p75
0,1927834,0.033451,0.567241,-3.071144,0.928856,-0.321144,0.078856,0.428856


In [40]:
# Per-item bias and Bayesian average from training data
# Bayesian avg = (C * global_mean + sum_of_ratings) / (C + n_reviews)
# where C = median item review count in training set (damping constant)

# Compute damping constant
DAMPING_C_RESULT = con.execute(f"""
    SELECT PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY cnt) AS median_item_count
    FROM (
        SELECT item_idx, COUNT(*) AS cnt
        FROM {ID_MAP_TABLE}
        WHERE split = 'train'
        GROUP BY item_idx
    )
""").fetchdf()

DAMPING_C = float(DAMPING_C_RESULT["median_item_count"].iloc[0])
print(f"Damping constant C (median train item count): {DAMPING_C}")

ITEM_BIAS_TABLE = "item_bias_stage9"

con.execute(f"DROP TABLE IF EXISTS {ITEM_BIAS_TABLE}")

con.execute(f"""
CREATE TABLE {ITEM_BIAS_TABLE} AS
SELECT
    item_idx,
    original_item_id,
    COUNT(*) AS train_review_count,
    AVG(rating) AS train_mean_rating,
    AVG(rating) - {GLOBAL_MEAN} AS item_bias,
    SUM(rating) AS sum_ratings,
    ({DAMPING_C} * {GLOBAL_MEAN} + SUM(rating)) / ({DAMPING_C} + COUNT(*)) AS bayesian_avg
FROM {ID_MAP_TABLE}
WHERE split = 'train'
GROUP BY item_idx, original_item_id
""")

print(f"Created item bias table: {ITEM_BIAS_TABLE}")

item_bias_stats = con.execute(f"""
SELECT
    COUNT(*) AS n_items,
    AVG(item_bias) AS mean_bias,
    STDDEV(item_bias) AS std_bias,
    MIN(item_bias) AS min_bias,
    MAX(item_bias) AS max_bias,
    AVG(bayesian_avg) AS mean_bayesian_avg,
    MIN(bayesian_avg) AS min_bayesian_avg,
    MAX(bayesian_avg) AS max_bayesian_avg
FROM {ITEM_BIAS_TABLE}
""").fetchdf()

display(item_bias_stats)

Damping constant C (median train item count): 18.0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Created item bias table: item_bias_stage9


,n_items,mean_bias,std_bias,min_bias,max_bias,mean_bayesian_avg,min_bayesian_avg,max_bayesian_avg
0,309395,-0.088553,0.654908,-3.071144,0.928856,4.035257,1.872657,4.948716


In [41]:
# Quick sanity: top 10 and bottom 10 items by Bayesian average
print("Top 10 items by Bayesian average (popularity baseline ranking):")
top_items = con.execute(f"""
    SELECT item_idx, train_review_count, train_mean_rating, bayesian_avg
    FROM {ITEM_BIAS_TABLE}
    ORDER BY bayesian_avg DESC
    LIMIT 10
""").fetchdf()
display(top_items)

print("\nBottom 10 items by Bayesian average:")
bottom_items = con.execute(f"""
    SELECT item_idx, train_review_count, train_mean_rating, bayesian_avg
    FROM {ITEM_BIAS_TABLE}
    ORDER BY bayesian_avg ASC
    LIMIT 10
""").fetchdf()
display(bottom_items)

Top 10 items by Bayesian average (popularity baseline ranking):


,item_idx,train_review_count,train_mean_rating,bayesian_avg
0,117566,815,4.968098,4.948716
1,79360,215,5.000000,4.928243
2,120011,287,4.979094,4.925510
3,79493,297,4.976431,4.924700
4,170062,334,4.964072,4.918411
5,52068,458,4.949782,4.916556
6,130458,974,4.929158,4.913589
7,47620,196,4.989796,4.912526
8,228949,214,4.981308,4.910692
9,79626,211,4.981043,4.909522



Bottom 10 items by Bayesian average:


,item_idx,train_review_count,train_mean_rating,bayesian_avg
0,48258,129,1.565891,1.872657
1,45876,70,1.400000,1.946370
2,91161,97,1.567010,1.958962
3,226997,93,1.580645,1.984510
4,48332,95,1.642105,2.029032
5,47638,235,1.902128,2.056445
6,37828,673,2.115899,2.166832
7,242600,90,1.788889,2.169265
8,53160,87,1.781609,2.174101
9,46083,98,1.836735,2.183453


## Stage 10 — Sub-rating Cleaning & Item Metadata

Clean out-of-scale sub-ratings (0 → null), then compute per-item sub-rating
averages from training data. Also extract geo_id from hotel URLs.

In [42]:
# STAGE 10 — Build item-side metadata from DEDUPED TRAIN interactions only
# This version avoids reintroducing duplicate user-item rows from the raw table.

STAGE10_SOURCE_TABLE = SPLIT_TABLE          # from Section 6 / 7
TRAIN_MATCH_TABLE = "train_raw_match_stage10"
ITEM_META_TABLE = "item_side_info_stage10"

In [43]:
# 1) Match each FINAL TRAIN interaction to raw review rows using the full interaction key.
#    Then keep only ONE raw row per final interaction_id.

con.execute(f"DROP TABLE IF EXISTS {TRAIN_MATCH_TABLE}")

con.execute(f"""
CREATE TABLE {TRAIN_MATCH_TABLE} AS
WITH train_interactions AS (
    SELECT
        interaction_id,
        user_id,
        item_id,
        rating,
        timestamp
    FROM {STAGE10_SOURCE_TABLE}
    WHERE split = 'train'
),
candidate_matches AS (
    SELECT
        t.interaction_id,
        t.user_id,
        t.item_id,
        t.rating,
        t.timestamp,
        r.title,
        r.text,
        r.property_dict,
        ROW_NUMBER() OVER (
            PARTITION BY t.interaction_id
            ORDER BY
                CASE WHEN r.property_dict IS NOT NULL THEN 1 ELSE 0 END DESC,
                LENGTH(COALESCE(r.text, '')) DESC,
                COALESCE(r.title, '') DESC
        ) AS rn
    FROM train_interactions t
    JOIN {RAW_TABLE} r
      ON t.user_id   = r.author
     AND t.item_id   = r.hotel_url
     AND t.rating    = r.rating
     AND t.timestamp = r.date
)
SELECT
    interaction_id,
    user_id,
    item_id,
    rating,
    timestamp,
    title,
    text,
    property_dict
FROM candidate_matches
WHERE rn = 1
""")

print(f"Created train-match table: {TRAIN_MATCH_TABLE}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Created train-match table: train_raw_match_stage10


In [44]:
# 2) Audit the match rate so you can verify how many train interactions were matched back
#    to a single raw review row.

stage10_match_audit_df = con.execute(f"""
WITH train_counts AS (
    SELECT COUNT(*) AS n_train_interactions
    FROM {STAGE10_SOURCE_TABLE}
    WHERE split = 'train'
),
matched_counts AS (
    SELECT COUNT(*) AS n_matched_rows
    FROM {TRAIN_MATCH_TABLE}
)
SELECT
    n_train_interactions,
    n_matched_rows,
    n_train_interactions - n_matched_rows AS unmatched_train_interactions,
    ROUND(100.0 * n_matched_rows / n_train_interactions, 4) AS matched_pct
FROM train_counts, matched_counts
""").fetchdf()

display(stage10_match_audit_df)

,n_train_interactions,n_matched_rows,unmatched_train_interactions,matched_pct
0,16064988,16064988,0,100.0


In [45]:
# 3) Build unified item metadata from ITEM_BIAS_TABLE (Stage 9) + matched train rows.
#    - Includes bias terms, Bayesian avg, geo_id, text coverage, and cleaned sub-ratings.
#    - Sub-ratings of 0 are excluded (treated as null) per Step 3 of the pipeline plan.

con.execute(f"DROP TABLE IF EXISTS {ITEM_META_TABLE}")

con.execute(f"""
CREATE TABLE {ITEM_META_TABLE} AS
SELECT
    b.item_idx,
    b.original_item_id AS item_id,
    REGEXP_EXTRACT(b.original_item_id, 'g(\\d+)', 1) AS geo_id,
    b.train_review_count,
    b.train_mean_rating,
    b.item_bias,
    b.bayesian_avg,

    -- text coverage
    COUNT(CASE WHEN m.title IS NOT NULL AND TRIM(m.title) <> '' THEN 1 END) AS n_reviews_with_title,
    COUNT(CASE WHEN m.text  IS NOT NULL AND TRIM(m.text)  <> '' THEN 1 END) AS n_reviews_with_text,
    COUNT(CASE WHEN m.property_dict IS NOT NULL THEN 1 END) AS n_reviews_with_any_subrating,

    -- cleaned sub-rating averages (0 values excluded as invalid)
    AVG(CASE WHEN m.property_dict.value > 0 THEN m.property_dict.value END) AS avg_value,
    AVG(CASE WHEN m.property_dict.rooms > 0 THEN m.property_dict.rooms END) AS avg_rooms,
    AVG(CASE WHEN m.property_dict.location > 0 THEN m.property_dict.location END) AS avg_location,
    AVG(CASE WHEN m.property_dict.cleanliness > 0 THEN m.property_dict.cleanliness END) AS avg_cleanliness,
    AVG(CASE WHEN m.property_dict.service > 0 THEN m.property_dict.service END) AS avg_service,
    AVG(CASE WHEN m.property_dict."sleep quality" > 0 THEN m.property_dict."sleep quality" END) AS avg_sleep_quality

FROM {ITEM_BIAS_TABLE} b
LEFT JOIN {TRAIN_MATCH_TABLE} m
    ON b.original_item_id = m.item_id
GROUP BY
    b.item_idx,
    b.original_item_id,
    b.train_review_count,
    b.train_mean_rating,
    b.item_bias,
    b.bayesian_avg
""")

print(f"Created item metadata table: {ITEM_META_TABLE}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Created item metadata table: item_side_info_stage10


In [46]:
# 4) Summary of the item metadata table

item_meta_summary_df = con.execute(f"""
SELECT
    COUNT(*) AS n_items,
    SUM(train_review_count) AS total_train_reviews,
    AVG(train_review_count) AS avg_reviews_per_item,

    -- bias / popularity stats
    AVG(item_bias) AS mean_item_bias,
    AVG(bayesian_avg) AS mean_bayesian_avg,

    -- coverage
    COUNT(geo_id) AS items_with_geo_id,
    COUNT(CASE WHEN n_reviews_with_title > 0 THEN 1 END) AS items_with_any_title,
    COUNT(CASE WHEN n_reviews_with_text > 0 THEN 1 END) AS items_with_any_text,
    COUNT(CASE WHEN n_reviews_with_any_subrating > 0 THEN 1 END) AS items_with_any_subrating,

    -- sub-rating availability
    COUNT(avg_value) AS items_with_avg_value,
    COUNT(avg_rooms) AS items_with_avg_rooms,
    COUNT(avg_location) AS items_with_avg_location,
    COUNT(avg_cleanliness) AS items_with_avg_cleanliness,
    COUNT(avg_service) AS items_with_avg_service,
    COUNT(avg_sleep_quality) AS items_with_avg_sleep_quality
FROM {ITEM_META_TABLE}
""").fetchdf()

display(item_meta_summary_df)

,n_items,total_train_reviews,avg_reviews_per_item,mean_item_bias,mean_bayesian_avg,items_with_geo_id,items_with_any_title,items_with_any_text,items_with_any_subrating,items_with_avg_value,items_with_avg_rooms,items_with_avg_location,items_with_avg_cleanliness,items_with_avg_service,items_with_avg_sleep_quality
0,309395,16064988.0,51.923877,-0.088553,4.035257,309395,309395,309395,309395,294341,292488,294174,294372,305862,291888


In [47]:
# 5) Build user metadata table (wraps Stage 9 bias table with a consistent name)

USER_META_TABLE = "user_meta_stage10"

con.execute(f"DROP TABLE IF EXISTS {USER_META_TABLE}")

con.execute(f"""
CREATE TABLE {USER_META_TABLE} AS
SELECT *
FROM {USER_BIAS_TABLE}
""")

print(f"Created user metadata table: {USER_META_TABLE}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Created user metadata table: user_meta_stage10


## Stage 11 — Export

Export all artifacts to disk:
- `interactions_model_ready.parquet` — primary interaction table with integer IDs and split labels
- `user_meta.parquet` / `item_meta.parquet` — metadata with bias terms, popularity scores, sub-ratings
- `user_id_mapping.parquet` / `item_id_mapping.parquet` — reversible string ↔ integer lookups
- `duplicate_features.parquet` — repeat-visit signal for Phase 2
- `split_config.json` — full reproducibility record of all pipeline parameters

In [48]:
# Export directory setup

EXPORT_DIR = r"D:\256_group_dataset\phase1_outputs"
os.makedirs(EXPORT_DIR, exist_ok=True)

def safe_path(p):
    """Convert Windows backslashes for DuckDB COPY statements"""
    return p.replace(chr(92), "/")

print("Export directory:", EXPORT_DIR)

Export directory: D:\256_group_dataset\phase1_outputs


In [49]:
# Build the final interaction export table with integer IDs

FINAL_EXPORT_TABLE = "interactions_model_ready_final"

con.execute(f"DROP TABLE IF EXISTS {FINAL_EXPORT_TABLE}")

con.execute(f"""
CREATE TABLE {FINAL_EXPORT_TABLE} AS
SELECT
    interaction_id,
    user_idx,
    item_idx,
    CAST(rating AS DOUBLE) AS rating,
    CAST(timestamp AS TIMESTAMP) AS timestamp,
    split,
    original_user_id,
    original_item_id
FROM {ID_MAP_TABLE}
""")

# Final audit
final_audit = con.execute(f"""
SELECT
    COUNT(*) AS n_interactions,
    COUNT(DISTINCT user_idx) AS n_users,
    COUNT(DISTINCT item_idx) AS n_items,
    MIN(rating) AS min_rating,
    MAX(rating) AS max_rating,
    MIN(timestamp) AS min_ts,
    MAX(timestamp) AS max_ts
FROM {FINAL_EXPORT_TABLE}
""").fetchdf()

print(f"Final export table: {FINAL_EXPORT_TABLE}")
display(final_audit)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Final export table: interactions_model_ready_final


,n_interactions,n_users,n_items,min_rating,max_rating,min_ts,max_ts
0,19920656,1927834,309904,1.0,5.0,2002-07-01,2019-09-20


In [50]:
# Export interactions: full file + per-split files

# Full dataset
full_parquet = os.path.join(EXPORT_DIR, "interactions_model_ready.parquet")
con.execute(f"""
COPY (
    SELECT interaction_id, user_idx, item_idx, rating, timestamp, split,
           original_user_id, original_item_id
    FROM {FINAL_EXPORT_TABLE}
    ORDER BY timestamp ASC, interaction_id ASC
) TO '{safe_path(full_parquet)}'
WITH (FORMAT PARQUET)
""")
print(f"Exported: {full_parquet}")

# Per-split files
for split_name in ["train", "valid", "test"]:
    split_path = os.path.join(EXPORT_DIR, f"interactions_model_ready_{split_name}.parquet")
    con.execute(f"""
    COPY (
        SELECT interaction_id, user_idx, item_idx, rating, timestamp, split,
               original_user_id, original_item_id
        FROM {FINAL_EXPORT_TABLE}
        WHERE split = '{split_name}'
        ORDER BY timestamp ASC, interaction_id ASC
    ) TO '{safe_path(split_path)}'
    WITH (FORMAT PARQUET)
    """)
    print(f"Exported: {split_path}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Exported: D:\256_group_dataset\phase1_outputs\interactions_model_ready.parquet


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Exported: D:\256_group_dataset\phase1_outputs\interactions_model_ready_train.parquet


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Exported: D:\256_group_dataset\phase1_outputs\interactions_model_ready_valid.parquet


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Exported: D:\256_group_dataset\phase1_outputs\interactions_model_ready_test.parquet


In [51]:
# Export ID mappings

user_map_path = os.path.join(EXPORT_DIR, "user_id_mapping.parquet")
item_map_path = os.path.join(EXPORT_DIR, "item_id_mapping.parquet")

con.execute(f"""
COPY (
    SELECT DISTINCT original_user_id, user_idx
    FROM {ID_MAP_TABLE}
    ORDER BY user_idx
) TO '{safe_path(user_map_path)}'
WITH (FORMAT PARQUET)
""")

con.execute(f"""
COPY (
    SELECT DISTINCT original_item_id, item_idx
    FROM {ID_MAP_TABLE}
    ORDER BY item_idx
) TO '{safe_path(item_map_path)}'
WITH (FORMAT PARQUET)
""")

print(f"Exported: {user_map_path}")
print(f"Exported: {item_map_path}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Exported: D:\256_group_dataset\phase1_outputs\user_id_mapping.parquet
Exported: D:\256_group_dataset\phase1_outputs\item_id_mapping.parquet


In [52]:
# Export metadata tables

user_meta_path = os.path.join(EXPORT_DIR, "user_meta.parquet")
item_meta_path = os.path.join(EXPORT_DIR, "item_meta.parquet")

con.execute(f"""
COPY (
    SELECT * FROM {USER_META_TABLE}
    ORDER BY user_idx
) TO '{safe_path(user_meta_path)}'
WITH (FORMAT PARQUET)
""")

con.execute(f"""
COPY (
    SELECT * FROM {ITEM_META_TABLE}
    ORDER BY item_idx
) TO '{safe_path(item_meta_path)}'
WITH (FORMAT PARQUET)
""")

print(f"Exported: {user_meta_path}")
print(f"Exported: {item_meta_path}")

Exported: D:\256_group_dataset\phase1_outputs\user_meta.parquet
Exported: D:\256_group_dataset\phase1_outputs\item_meta.parquet


In [53]:
# Export duplicate features table for Phase 2 use
# (repeat-visit signal: visit count, rating change, time gap)

dup_features_path = os.path.join(EXPORT_DIR, "duplicate_features.parquet")

con.execute(f"""
COPY (
    SELECT
        d.user_id AS original_user_id,
        d.item_id AS original_item_id,
        um.user_idx,
        im.item_idx,
        d.pair_count AS visit_count,
        CASE WHEN d.min_rating != d.max_rating THEN TRUE ELSE FALSE END AS rating_changed,
        DATEDIFF('day', d.first_timestamp, d.last_timestamp) AS time_gap_days,
        d.min_rating,
        d.max_rating,
        d.first_timestamp,
        d.last_timestamp
    FROM {DUP_GROUPS_TABLE} d
    LEFT JOIN (
        SELECT DISTINCT original_user_id, user_idx FROM {ID_MAP_TABLE}
    ) um ON d.user_id = um.original_user_id
    LEFT JOIN (
        SELECT DISTINCT original_item_id, item_idx FROM {ID_MAP_TABLE}
    ) im ON d.item_id = im.original_item_id
    WHERE um.user_idx IS NOT NULL AND im.item_idx IS NOT NULL
    ORDER BY um.user_idx, im.item_idx
) TO '{safe_path(dup_features_path)}'
WITH (FORMAT PARQUET)
""")

print(f"Exported: {dup_features_path}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Exported: D:\256_group_dataset\phase1_outputs\duplicate_features.parquet


In [54]:
# Export pipeline configuration for reproducibility

import json

split_config = {
    "pipeline_version": "1.0",
    "source_file": FILE_PATH,
    "k_core": K_CORE_K,
    "split_strategy": "per_user_leave_last_out",
    "split_description": "For each user: most recent interaction -> test, second most recent -> valid, all others -> train",
    "sentinel_users_removed": PLACEHOLDER_USER_IDS,
    "deduplication": "keep_most_recent_per_user_item_pair",
    "sub_rating_cleaning": "values_of_0_set_to_null",
    "global_mean_train": round(GLOBAL_MEAN, 6),
    "damping_constant_C": DAMPING_C,
    "n_users": int(id_map_audit_df.iloc[0]["n_user_ids"]),
    "n_items": int(id_map_audit_df.iloc[0]["n_item_ids"]),
    "n_interactions": int(final_audit.iloc[0]["n_interactions"]),
    "n_train": int(con.execute(f"SELECT COUNT(*) FROM {FINAL_EXPORT_TABLE} WHERE split='train'").fetchone()[0]),
    "n_valid": int(con.execute(f"SELECT COUNT(*) FROM {FINAL_EXPORT_TABLE} WHERE split='valid'").fetchone()[0]),
    "n_test": int(con.execute(f"SELECT COUNT(*) FROM {FINAL_EXPORT_TABLE} WHERE split='test'").fetchone()[0]),
}

config_path = os.path.join(EXPORT_DIR, "split_config.json")
with open(config_path, "w") as f:
    json.dump(split_config, f, indent=2)

print(f"Exported: {config_path}")
print(json.dumps(split_config, indent=2))

Exported: D:\256_group_dataset\phase1_outputs\split_config.json
{
  "pipeline_version": "1.0",
  "source_file": "D:\\256_group_dataset\\HotelRec.txt",
  "k_core": 5,
  "split_strategy": "per_user_leave_last_out",
  "split_description": "For each user: most recent interaction -> test, second most recent -> valid, all others -> train",
  "sentinel_users_removed": [
    "/undefined",
    "undefined",
    "null",
    "none",
    "nan"
  ],
  "deduplication": "keep_most_recent_per_user_item_pair",
  "sub_rating_cleaning": "values_of_0_set_to_null",
  "global_mean_train": 4.071144,
  "damping_constant_C": 18.0,
  "n_users": 1927834,
  "n_items": 309904,
  "n_interactions": 19920656,
  "n_train": 16064988,
  "n_valid": 1927834,
  "n_test": 1927834
}


In [55]:
# Updated data dictionary

data_dictionary = pd.DataFrame([
    {"column_name": "interaction_id", "dtype": "integer", "description": "Unique row identifier carried through preprocessing"},
    {"column_name": "user_idx", "dtype": "integer", "description": "Contiguous integer user ID (0 to n_users-1)"},
    {"column_name": "item_idx", "dtype": "integer", "description": "Contiguous integer item ID (0 to n_items-1)"},
    {"column_name": "rating", "dtype": "float", "description": "Explicit overall hotel rating on the 1-5 scale"},
    {"column_name": "timestamp", "dtype": "timestamp", "description": "Review timestamp used for temporal split ordering"},
    {"column_name": "split", "dtype": "string", "description": "Per-user leave-last-out split: train, valid, or test"},
    {"column_name": "original_user_id", "dtype": "string", "description": "Original TripAdvisor author username"},
    {"column_name": "original_item_id", "dtype": "string", "description": "Original TripAdvisor hotel URL"},
])

data_dict_path = os.path.join(EXPORT_DIR, "data_dictionary.csv")
data_dictionary.to_csv(data_dict_path, index=False)

display(data_dictionary)
print(f"\nExported: {data_dict_path}")

,column_name,dtype,description
0,interaction_id,integer,Unique row identifier carried through preproce...
1,user_idx,integer,Contiguous integer user ID (0 to n_users-1)
2,item_idx,integer,Contiguous integer item ID (0 to n_items-1)
3,rating,float,Explicit overall hotel rating on the 1-5 scale
4,timestamp,timestamp,Review timestamp used for temporal split ordering
5,split,string,"Per-user leave-last-out split: train, valid, o..."
6,original_user_id,string,Original TripAdvisor author username
7,original_item_id,string,Original TripAdvisor hotel URL



Exported: D:\256_group_dataset\phase1_outputs\data_dictionary.csv


In [56]:
# Final summary of all exported files

print("=" * 70)
print("PIPELINE COMPLETE - Exported files:")
print("=" * 70)

import glob
for f in sorted(glob.glob(os.path.join(EXPORT_DIR, "*"))):
    size_mb = os.path.getsize(f) / (1024 * 1024)
    print(f"  {os.path.basename(f):50s} {size_mb:8.1f} MB")

print(f"\nTotal files: {len(glob.glob(os.path.join(EXPORT_DIR, '*')))}")

PIPELINE COMPLETE - Exported files:
  data_dictionary.csv                                     0.0 MB
  duplicate_features.parquet                             36.2 MB
  interactions_model_ready.csv                         2765.5 MB
  interactions_model_ready.parquet                      983.3 MB
  interactions_model_ready_data_dictionary.csv            0.0 MB
  interactions_model_ready_test.csv                     276.5 MB
  interactions_model_ready_test.parquet                 119.9 MB
  interactions_model_ready_train.csv                   2211.0 MB
  interactions_model_ready_train.parquet                818.6 MB
  interactions_model_ready_valid.csv                    277.9 MB
  interactions_model_ready_valid.parquet                124.9 MB
  item_id_mapping.parquet                                 9.9 MB
  item_meta.parquet                                      19.0 MB
  split_config.json                                       0.0 MB
  user_id_mapping.parquet                             